# Exp2 Classificação sMCI vs pMCI
- analise demográfica

In [29]:
import pandas as pd

INPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI.txt"
OUTPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI_extremos.csv"

df = pd.read_csv(INPUT_PATH)
df["MRI_DATE"] = pd.to_datetime(df["MRI_DATE"])

parts = []
for id_pt, g in df.groupby("ID_PT", sort=False):
    g_smci = g.loc[g["GROUP"] == "sMCI"].sort_values("MRI_DATE", ascending=True)
    early_smci = g_smci.head(3).copy()

    g_pmci = g.loc[g["GROUP"] == "pMCI"].sort_values("MRI_DATE", ascending=True)
    late_pmci = g_pmci.tail(3).copy()

    parts.extend([early_smci, late_pmci])

df_ext = pd.concat(parts, ignore_index=True)
df_ext.to_csv(OUTPUT_PATH, index=False)

print("Salvo:" , OUTPUT_PATH)
print("df_ext:", df_ext.shape, "linhas |", df_ext["ID_PT"].nunique(), "pacientes")
df_ext.head(10)

Salvo: /mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI_extremos.csv
df_ext: (1575, 7) linhas | 525 pacientes


,ID_PT,ID_IMG,DIAG,SEX,AGE,MRI_DATE,GROUP
0,002_S_0729,I19058,MCI,F,65,2006-07-17,pMCI
1,002_S_0729,I41585,MCI,F,66,2007-02-22,pMCI
2,002_S_0729,I71848,AD,F,66,2007-09-05,pMCI
3,002_S_0782,I20520,MCI,M,82,2006-08-14,sMCI
4,002_S_0782,I48752,MCI,M,82,2007-04-11,sMCI
5,002_S_0782,I73816,MCI,M,83,2007-09-19,sMCI
6,002_S_0954,I26139,MCI,F,69,2006-10-10,pMCI
7,002_S_0954,I53469,MCI,F,70,2007-05-03,pMCI
8,002_S_0954,I77803,AD,F,70,2007-10-15,pMCI
9,002_S_1070,I56575,MCI,M,74,2007-06-07,pMCI


In [30]:
import pandas as pd

# df_ext já carregado; se não:
# df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 525
Linhas totais: 1575
Checagem linhas/3: 525.0

Por GROUP:
GROUP
pMCI    128
sMCI    397
Name: count, dtype: int64

Por SEX:
SEX
F    222
M    303
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
pMCI    54   74  128
sMCI   168  229  397
All    222  303  525


In [32]:
import pandas as pd

adnimerge = pd.read_csv("csvs/adnimerged.csv")
image_data = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

In [33]:
cols = ["ID_IMG", "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE"]  # etc.
adni_sub = (
    adnimerge[cols]
    .drop_duplicates(subset=["ID_IMG"], keep="last")
)
image_data = image_data.merge(adni_sub, on="ID_IMG", how="left")

In [34]:
image_data.head()

,ID_PT,ID_IMG,DIAG,SEX,AGE,MRI_DATE,GROUP,MMSE_SCORE,CDR_GLOBAL,ADAS_SCORE,FAQ_SCORE
0,002_S_0729,I19058,MCI,F,65,2006-07-17,pMCI,27.0,0.5,6.67,7.0
1,002_S_0729,I41585,MCI,F,66,2007-02-22,pMCI,27.0,0.5,8.00,5.0
2,002_S_0729,I71848,AD,F,66,2007-09-05,pMCI,23.0,0.5,11.67,9.0
3,002_S_0782,I20520,MCI,M,82,2006-08-14,sMCI,29.0,0.5,16.33,0.0
4,002_S_0782,I48752,MCI,M,82,2007-04-11,sMCI,30.0,0.5,12.00,2.0


In [35]:
image_data.groupby("GROUP")["ID_PT"].nunique()

GROUP
pMCI    128
sMCI    397
Name: ID_PT, dtype: int64

In [36]:
dist_pacientes = (
    image_data.drop_duplicates(subset=["ID_PT", "GROUP"])
    .groupby(["GROUP", "SEX"])["ID_PT"]
    .nunique()
    .unstack(fill_value=0)
)

dist_pacientes

SEX,F,M
GROUP,,
pMCI,54,74
sMCI,168,229


In [37]:
pts = image_data.groupby(["GROUP", "ID_PT"], as_index=False)["AGE"].min()

for g in ["sMCI", "pMCI"]:
    a = pts.loc[pts["GROUP"] == g, "AGE"]
    print(
        f"{g}: n={a.count()}, "
        f"min={a.min():.1f}, max={a.max():.1f}, "
        f"media={a.mean():.2f}, desvio={a.std():.2f}"
    )

sMCI: n=397, min=55.0, max=91.0, media=73.52, desvio=7.48
pMCI: n=128, min=57.0, max=89.0, media=75.04, desvio=7.01


In [19]:
pts = (
    image_data.sort_values(["GROUP", "ID_PT", "MRI_DATE"])
    .drop_duplicates(subset=["GROUP", "ID_PT"], keep="first")
)

pts.groupby("GROUP")[["MMSE_SCORE", "FAQ_SCORE"]].agg(["mean", "std", "count"])

MMSE_SCORE                 FAQ_SCORE                
            mean       std count      mean       std count
GROUP                                                     
pMCI   26.039062  2.249768   128  6.453125  4.619131   128
sMCI   27.758186  1.822132   397  2.503778  3.573628   397

# Uso de GPU

In [1]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

I0000 00:00:1778947719.611291 1351961 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 0 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
GPU 1 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
